In [ ]:
%pip install -r requirements.txt

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

path_1 = "../data/outlier_deal_for_nan_exp1.csv"
path_2 = "../data/outlier_deal_for_nan_exp2.csv"
df_1 = pd.read_csv(path_1)
df_2 = pd.read_csv(path_2)

#### 交互作用

In [4]:
def make_fc(df_src):
    return (
        df_src.sort_values(["responseId", "frame_block", "ref_subblock"])
              .groupby(["responseId", "frame_block"], as_index=False)[
                  ["frame_choice", "Message Group"]
              ]
              .first()          # ブロック内の最初の非NaN値を採用
              .dropna()         # 5行とも欠損のブロックは除く
    )

In [5]:
d = pd.concat(
    [make_fc(df_1).assign(ref_cond="なし"),
     make_fc(df_2).assign(ref_cond="あり")],
    ignore_index=True,
).rename(columns={"Message Group": "group"})

d["frame_choice"] = d["frame_choice"].astype(int)
d["group"] = pd.Categorical(d["group"], categories=["neutral", "negative", "positive"])

# 確認: 1人あたり最大20ブロック、値は 0/1 のみ、群は3つのみ
print(d.groupby("ref_cond")["responseId"].nunique())
print(d.groupby(["ref_cond", "responseId"]).size().groupby("ref_cond").describe())
print(d["frame_choice"].value_counts())
print(d["group"].value_counts(dropna=False))     # NaN が出たら、群の表記が違う

m = smf.gee(
    "frame_choice ~ C(group) * C(ref_cond)",
    groups="responseId", data=d,
    family=sm.families.Binomial(),
    cov_struct=sm.cov_struct.Exchangeable(),
).fit()
print(m.wald_test_terms())    # 感情・参照点・交互作用

ref_cond
あり    88
なし    88
Name: responseId, dtype: int64
          count       mean       std   min   25%   50%   75%   max
ref_cond                                                          
あり         88.0  19.863636  0.433691  18.0  20.0  20.0  20.0  20.0
なし         88.0  19.818182  0.578254  16.0  20.0  20.0  20.0  20.0
frame_choice
1    2082
0    1410
Name: count, dtype: int64
group
neutral     1230
positive    1168
negative    1094
Name: count, dtype: int64
                          chi2    P>chi2  df constraint
Intercept             0.713315  0.398345              1
C(group)              6.150016  0.046189              2
C(ref_cond)           1.174528  0.278473              1
C(group):C(ref_cond)  0.889407  0.641014              2


In [6]:
m = smf.gee(
    "frame_choice ~ C(group) * C(ref_cond)",
    groups="responseId", data=d,
    family=sm.families.Binomial(),
    cov_struct=sm.cov_struct.Exchangeable(),
).fit()

inter = [k for k in m.params.index if ":" in k]
ci = m.conf_int().loc[inter]
print(pd.DataFrame({
    "OR比": np.exp(m.params[inter]),
    "lower": np.exp(ci[0]),
    "upper": np.exp(ci[1]),
    "p": m.pvalues[inter],
}).round(3))

                                          OR比  lower  upper      p
C(group)[T.negative]:C(ref_cond)[T.なし]  1.349  0.474  3.837  0.574
C(group)[T.positive]:C(ref_cond)[T.なし]  0.790  0.298  2.097  0.636


#### 主効果

In [8]:
m_add = smf.gee(
    "frame_choice ~ C(group) + C(ref_cond)",
    groups="responseId", data=d,
    family=sm.families.Binomial(),
    cov_struct=sm.cov_struct.Exchangeable(),
).fit()

print(m_add.wald_test_terms())    # 感情の主効果・参照点の主効果(交互作用なしなので、そのまま主効果)

ci = m_add.conf_int()
print(pd.DataFrame({
    "OR": np.exp(m_add.params),
    "lower": np.exp(ci[0]),
    "upper": np.exp(ci[1]),
    "p": m_add.pvalues,
}).round(4))

# 条件 × 群ごとの half empty の割合(図・表用)
print(d.groupby(["ref_cond", "group"], observed=True)["frame_choice"]
        .agg(割合="mean", ブロック数="size").round(3))

                  chi2    P>chi2  df constraint
Intercept     0.922302  0.336871              1
C(group)     19.270118  0.000065              2
C(ref_cond)   2.517001  0.112625              1
                          OR   lower   upper       p
Intercept             1.1882  0.8357  1.6893  0.3369
C(group)[T.negative]  2.1448  1.2732  3.6129  0.0041
C(group)[T.positive]  0.6170  0.3792  1.0040  0.0519
C(ref_cond)[T.なし]     1.4065  0.9228  2.1437  0.1126
                      割合  ブロック数
ref_cond group                 
あり       neutral   0.542    635
         negative  0.686    497
         positive  0.450    616
なし       neutral   0.629    595
         negative  0.807    597
         positive  0.478    552


In [9]:
tt = m_add.t_test("C(group)[T.positive] - C(group)[T.negative] = 0")
print(tt.summary())
est = float(np.squeeze(tt.effect))
lo, hi = np.squeeze(tt.conf_int())
print("OR =", np.exp(est).round(3), "95%CI", np.exp([lo, hi]).round(3), "p =", float(np.squeeze(tt.pvalue)))

                             Test for Constraints                             
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
c0            -1.2459      0.285     -4.377      0.000      -1.804      -0.688
OR = 0.288 95%CI [0.165 0.503] p = 1.2006968179350315e-05
